# 02 · Who responds? Heterogeneous effects on the Hillstrom email test

**Question.** Notebook 01 showed that both emails raised visits, purchases and spend on average. Does the effect differ across customers, and can a model find the customers who respond most, well enough to be useful on customers it has not seen?

**Data.** The same Hillstrom experiment (64,000 customers, three randomised arms). Each email is compared with the no-email control as a separate two-arm problem, so there are two treatments: men's email versus control, and women's email versus control.

**Approach**

1. Before any modelling, look for heterogeneity with pre-specified subgroups and a joint interaction test.
2. Build the evaluation machinery (Qini-style gain curves), check it against scikit-uplift, and test whether the whole pipeline can recover a planted effect in simulated data. A model comparison is only informative if the pipeline can find heterogeneity when it exists.
3. Compare four learners (S, T and X meta-learners and a causal forest) against a simple rule, on held-out customers, over repeated random splits.
4. Put uncertainty on the held-out results with a bootstrap, and compare them with a permutation null.
5. Check calibration by decile, and see what the causal forest relies on.

**Analysis plan**

- Primary outcome for judging models: **visit** (the best-powered outcome, from notebook 01). Conversion and spend are reported as secondary and expected to be noisy.
- Evaluation always uses customers the model was not trained on: a random 50/50 split within each two-arm dataset, stratified by arm, repeated five times.
- The headline metric is the **area between the model's gain curve and the random-targeting line** (defined in section 2), in units of extra outcome per customer. Positive means the model ranks responders ahead of non-responders better than chance.
- Subgroups are fixed in advance (listed in section 1), with a Holm correction across the subgroup tests.
- Whether targeting beats emailing everyone depends on what an email costs. That is notebook 04. This notebook only asks whether the ranking is better than random.

In [ ]:
import os, platform, warnings
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import confint_proportions_2indep

import lightgbm
import econml
import sklearn
from econml.dml import CausalForestDML
from econml.metalearners import SLearner, TLearner, XLearner
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklift.metrics import uplift_curve

warnings.filterwarnings("ignore")

SEED = 20261002
N_SPLITS = 5
N_BOOT = 1000          # bootstrap and permutation replicates on the first split
N_JOBS = min(4, os.cpu_count() or 1)
ALPHA = 0.05

DATA_PATH = Path("../data/raw/hillstrom.csv")
TABLE_DIR = Path("../reports/tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | statsmodels {statsmodels.__version__}")
print(f"lightgbm {lightgbm.__version__} | econml {econml.__version__} | scikit-learn {sklearn.__version__}")

## Load and prepare

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Data not found. Run 00_data_acquisition.ipynb first.")

df = pd.read_csv(DATA_PATH)
df["zip_code"] = df["zip_code"].replace({"Surburban": "Suburban"})   # label typo in the source file
assert len(df) == 64_000 and df.isna().sum().sum() == 0

# Pre-treatment features for the models. history is right-skewed, so it enters on the log scale.
FEATURE_SOURCE = ["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]
X_all = pd.get_dummies(df[FEATURE_SOURCE], columns=["zip_code", "channel"], dtype=float)
X_all["history"] = np.log(X_all["history"])
FEATURES = list(X_all.columns)

OUTCOMES = ["visit", "conversion", "spend"]
ARMS = {"mens_email": "Mens E-Mail", "womens_email": "Womens E-Mail"}
ARM_TITLE = {"mens_email": "men's email", "womens_email": "women's email"}

# Two-arm datasets: each email arm against the no-email control.
data = {}
for arm, label in ARMS.items():
    mask = df["segment"].isin([label, "No E-Mail"]).to_numpy()
    data[arm] = {
        "X": X_all[mask].reset_index(drop=True),
        "raw": df[mask].reset_index(drop=True),
        "t": (df.loc[mask, "segment"] == label).astype(int).to_numpy(),
        "y": {o: df.loc[mask, o].to_numpy(dtype=float) for o in OUTCOMES},
    }
    d = data[arm]
    print(f"{arm:13s} n = {len(d['t']):,}  treated = {d['t'].sum():,}  control = {(1 - d['t']).sum():,}")
print("Features:", FEATURES)

## 1 · Is there anything to find? Pre-specified subgroups and a joint test

Before fitting flexible models, a plain question: do the email effects differ across the groups a marketing team would think of first? The subgroup variables below were fixed before looking at any heterogeneity results.

| Variable | Groups |
|---|---|
| `newbie` | new customer in the last 12 months, or not |
| `channel` | phone, web, multichannel |
| `zip_code` | urban, suburban, rural |
| `mens`, `womens` | bought men's / women's merchandise in the past year, or not |
| `recency_group` | 1-3, 4-8, 9-12 months since last purchase |
| `history_tercile` | lowest, middle, highest third of past-year spend |

For each variable and each email, a Wald test (heteroskedasticity-robust) asks whether the effect on **visit** differs across its groups. With 14 such tests, p-values are adjusted with the Holm method. The effect table shows the visit effect within each group, with Newcombe intervals.

In [ ]:
df["recency_group"] = pd.cut(df["recency"], [0, 3, 8, 12], labels=["1-3", "4-8", "9-12"]).astype(str)
df["history_tercile"] = pd.qcut(df["history"], 3, labels=["low", "middle", "high"]).astype(str)
SUBGROUPS = ["newbie", "channel", "zip_code", "mens", "womens", "recency_group", "history_tercile"]

rows_effect, rows_test = [], []
for arm, label in ARMS.items():
    sub = df[df["segment"].isin([label, "No E-Mail"])].copy()
    sub["t"] = (sub["segment"] == label).astype(int)
    for var in SUBGROUPS:
        # effect within each group (visit)
        for level, g in sub.groupby(var):
            tr, ct = g[g["t"] == 1]["visit"], g[g["t"] == 0]["visit"]
            lo, hi = confint_proportions_2indep(int(tr.sum()), len(tr), int(ct.sum()), len(ct), method="newcomb", compare="diff")
            rows_effect.append({"arm": arm, "variable": var, "level": str(level), "n_treated": len(tr), "n_control": len(ct),
                                "effect": tr.mean() - ct.mean(), "ci_low": lo, "ci_high": hi})
        # Wald test of effect differences across the groups of this variable
        D = pd.get_dummies(sub[var].astype(str), drop_first=True, dtype=float)
        design = pd.concat([pd.Series(1.0, index=sub.index, name="const"), sub["t"].astype(float), D, D.mul(sub["t"], axis=0).add_suffix(":t")], axis=1)
        fit = sm.OLS(sub["visit"], design).fit(cov_type="HC3")
        inter_cols = [c for c in design.columns if c.endswith(":t")]
        R = np.zeros((len(inter_cols), design.shape[1]))
        for i, c in enumerate(inter_cols):
            R[i, list(design.columns).index(c)] = 1
        w = fit.wald_test(R, use_f=False, scalar=True)
        rows_test.append({"arm": arm, "variable": var, "df": len(inter_cols), "chi2": float(w.statistic), "p_raw": float(w.pvalue)})

effects = pd.DataFrame(rows_effect)
tests = pd.DataFrame(rows_test)
tests["p_holm"] = multipletests(tests["p_raw"], method="holm")[1]
tests["significant_holm"] = tests["p_holm"] < ALPHA
display(tests.round({"chi2": 2, "p_raw": 4, "p_holm": 4}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 6.2), sharey=True)
order = [(v, l) for v in SUBGROUPS for l in effects[effects["variable"] == v]["level"].unique()]
ypos = {k: i for i, k in enumerate(order)}
for ax, (arm, label) in zip(axes, ARMS.items()):
    e = effects[effects["arm"] == arm]
    y = [ypos[(v, l)] for v, l in zip(e["variable"], e["level"])]
    ax.errorbar(100 * e["effect"], y, xerr=[100 * (e["effect"] - e["ci_low"]), 100 * (e["ci_high"] - e["effect"])], fmt="o", capsize=3, ms=4)
    avg = df[df["segment"] == label]["visit"].mean() - df[df["segment"] == "No E-Mail"]["visit"].mean()
    ax.axvline(100 * avg, color="grey", ls="--", lw=1)
    ax.set_title(f"{ARM_TITLE[arm]} vs control")
    ax.set_xlabel("effect on visit (percentage points, 95% CI)")
axes[0].set_yticks(range(len(order)), [f"{v}: {l}" for v, l in order])
axes[0].invert_yaxis()
plt.suptitle("Visit effect by pre-specified subgroup (dashed line = overall effect)", y=1.0)
plt.tight_layout()
plt.show()

The subgroup tests look at one variable at a time. A joint test asks the broader question: do **all** the features together explain any variation in the effect? It fits a linear model of the outcome on treatment, the centred features and their interactions with treatment, and tests all interaction terms at once. Run on all three outcomes and both emails, with a Holm correction across the six tests.

In [ ]:
rows = []
for arm in ARMS:
    d = data[arm]
    Xc = d["X"] - d["X"].mean()
    Xc = Xc.drop(columns=[c for c in Xc.columns if c.endswith(("_Rural", "_Phone"))])   # drop one dummy per factor
    T = pd.Series(d["t"].astype(float), name="t")
    design = pd.concat([pd.Series(1.0, index=Xc.index, name="const"), T, Xc, Xc.mul(T, axis=0).add_suffix(":t")], axis=1)
    inter = [c for c in design.columns if c.endswith(":t")]
    R = np.zeros((len(inter), design.shape[1]))
    for i, c in enumerate(inter):
        R[i, list(design.columns).index(c)] = 1
    for o in OUTCOMES:
        fit = sm.OLS(d["y"][o], design).fit(cov_type="HC3")
        w = fit.wald_test(R, use_f=False, scalar=True)
        rows.append({"arm": arm, "outcome": o, "df": len(inter), "chi2": float(w.statistic), "p_raw": float(w.pvalue)})

joint = pd.DataFrame(rows)
joint["p_holm"] = multipletests(joint["p_raw"], method="holm")[1]
joint["significant_holm"] = joint["p_holm"] < ALPHA
display(joint.round({"chi2": 2, "p_raw": 4, "p_holm": 4}))

## 2 · The evaluation framework

**Gain curve.** Rank held-out customers from the highest predicted effect to the lowest. For each fraction φ of customers (from the top of the ranking), compare the average outcome of the emailed and the not-emailed customers among that top φ. That gives an estimated effect τ̂(φ) for the targeted group, and the **gain** is G(φ) = φ · τ̂(φ): the extra outcome per customer in the whole population if only the top φ were emailed. It is valid because assignment was randomised, so the emailed and not-emailed customers within any ranked group are comparable.

- G(1) is the overall effect of emailing everyone.
- **Random targeting** gives the straight line φ · G(1).
- A good model's curve rises above that line early. The area between the two is the headline metric, in extra outcome per customer (for visit, extra visits per customer). Call it **AUUC** (area under the uplift curve, relative to random).
- **Top-20% effect:** the observed average effect among the 20% ranked highest, to compare with the overall effect.

Ties in the ranking (for example, a rule based on a 12-level variable) are broken randomly, once, so that no tie order is accidentally favourable.

In [ ]:
def ranked_arrays(y, t, score, rng):
    """Order rows from highest to lowest score, with random tie-breaking."""
    order = np.lexsort((rng.random(len(y)), -score))
    return y[order], t[order]

def gain_curve(y, t, w=None):
    """Gain curve for rows already in ranked order. w = optional bootstrap weights.
    Returns (phi, G): share of customers targeted and extra outcome per customer in the whole population."""
    w = np.ones(len(y)) if w is None else w
    wt, wc = w * t, w * (1 - t)
    nt, nc = np.cumsum(wt), np.cumsum(wc)
    yt, yc = np.cumsum(wt * y), np.cumsum(wc * y)
    ok = (nt > 0) & (nc > 0)
    tau = np.zeros(len(y))
    tau[ok] = yt[ok] / nt[ok] - yc[ok] / nc[ok]
    phi = np.cumsum(w) / w.sum()
    return np.concatenate([[0.0], phi]), np.concatenate([[0.0], tau * phi])

def auuc(phi, G):
    """Area between the gain curve and the random-targeting line."""
    return np.trapezoid(G - phi * G[-1], phi)

def top_effect(phi, G, share=0.2):
    """Observed average effect among the top `share` of customers."""
    return np.interp(share, phi, G) / share

def evaluate(y, t, score, rng, w=None):
    ys, ts = ranked_arrays(y, t, score, rng)
    phi, G = gain_curve(ys, ts, w)
    return auuc(phi, G), top_effect(phi, G), G[-1]

### Check 1: the gain curve against scikit-uplift

scikit-uplift's `uplift_curve` computes the same quantity in customer counts. On a continuous random score (no ties) the two should agree to floating-point precision. Weights of 1 reproduce the unweighted curve.

In [ ]:
d = data["mens_email"]
test_idx = train_test_split(np.arange(len(d["t"])), test_size=0.5, stratify=d["t"], random_state=SEED)[1]
y_chk, t_chk = d["y"]["visit"][test_idx], d["t"][test_idx]
score_chk = np.random.default_rng(1).normal(size=len(test_idx))

ys, ts = ranked_arrays(y_chk, t_chk, score_chk, np.random.default_rng(0))
phi, G = gain_curve(ys, ts)
_, y_sklift = uplift_curve(y_chk, score_chk, t_chk)
max_diff = np.abs(G * len(y_chk) - y_sklift).max()
print(f"max |difference| between this gain curve and scikit-uplift: {max_diff:.2e}")
assert max_diff < 1e-8

phi_w, G_w = gain_curve(ys, ts, np.ones(len(ys)))
assert np.allclose(G, G_w) and np.allclose(phi, phi_w)
print("weights of 1 reproduce the unweighted curve")

### The models

All four learners use gradient-boosted trees (LightGBM) with deliberately conservative settings, because effect signals in marketing data are weak and flexible models overfit them. Binary outcomes are fitted with a squared-error objective, so predictions are on the probability scale. Because assignment was randomised, the propensity of treatment is known and constant within each two-arm dataset, so no propensity model is learned.

- **S-learner:** one outcome model with treatment as an input. The effect is the difference between predicting with and without treatment.
- **T-learner:** separate outcome models for emailed and not-emailed customers; the effect is the difference of their predictions.
- **X-learner:** the T-learner followed by a second stage that models the imputed individual effects.
- **Causal forest:** an honest forest (EconML `CausalForestDML`) grown to split on differences in treatment effect. It first removes outcome variation explained by the features (cross-fitted).
- **Rule:** the best single-variable ranking among recency (high or low first) and past-year spend (high or low first), with the direction chosen on the training half only.

LightGBM is run in its deterministic mode, so results do not depend on the number of threads.

In [ ]:
def make_lgb(seed):
    return LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=8, min_child_samples=200,
                         subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0,
                         deterministic=True, force_col_wise=True, n_jobs=N_JOBS, verbose=-1, random_state=seed)

def make_learner(name, seed):
    if name == "S-learner":
        return SLearner(overall_model=make_lgb(seed))
    if name == "T-learner":
        return TLearner(models=[make_lgb(seed), make_lgb(seed + 1)])
    if name == "X-learner":
        return XLearner(models=[make_lgb(seed), make_lgb(seed + 1)], cate_models=[make_lgb(seed + 2), make_lgb(seed + 3)],
                        propensity_model=DummyClassifier(strategy="prior"))
    if name == "Causal forest":
        return CausalForestDML(model_y=make_lgb(seed), model_t=DummyClassifier(strategy="prior"), discrete_treatment=True,
                               n_estimators=200, min_samples_leaf=100, max_samples=0.4, cv=3, random_state=seed, n_jobs=N_JOBS)
    raise ValueError(name)

LEARNERS = ["S-learner", "T-learner", "X-learner", "Causal forest"]
MODELS = LEARNERS + ["Rule"]

def rule_scores(X, y, t, rng):
    """Pick the best single-variable ranking on training data; return a scoring function for new data."""
    candidates = {"recency (recent first)": lambda Z: -Z["recency"].to_numpy(),
                  "recency (lapsed first)": lambda Z: Z["recency"].to_numpy(),
                  "past spend (high first)": lambda Z: Z["history"].to_numpy(),
                  "past spend (low first)": lambda Z: -Z["history"].to_numpy()}
    best = max(candidates, key=lambda k: evaluate(y, t, candidates[k](X), rng)[0])
    return best, candidates[best]

### Check 2: can this pipeline find heterogeneity when it exists?

A comparison that finds nothing is only informative if the method would have found something that was there. This check plants a known effect pattern in semi-synthetic data and runs the same learners and the same evaluation.

- **Features and treatment** are the real men's-email dataset (real covariates, real randomised assignment).
- **Outcome** is simulated: a baseline visit probability of 10.6% for everyone, plus an effect of +7.6 points on average (the size observed in notebook 01 for the men's email) that varies with the features. Effects range from roughly +2 to +16 points, higher for recent customers and for multichannel customers.
- The true individual effect is known, so the models' predictions can be compared with it directly.

If the learners recover this pattern, a null result on the real data means there is little heterogeneity of this size to find. If they cannot, the real-data comparison should not be read as evidence either way.

In [ ]:
d = data["mens_email"]
Xs, ts = d["X"], d["t"]
true_tau = (0.04
            + 0.05 * (Xs["recency"].to_numpy() <= 3)
            + 0.03 * Xs["channel_Multichannel"].to_numpy()
            + 0.015 * Xs["newbie"].to_numpy()
            - 0.004 * (Xs["recency"].to_numpy() - 6).clip(0, None)).clip(0.0, None)
true_tau = true_tau * 0.0763 / true_tau.mean()           # rescale so the average matches the observed +7.6 points
sim_rng = np.random.default_rng(SEED)
p_sim = np.clip(0.1062 + ts * true_tau, 0, 1)
y_sim = (sim_rng.random(len(ts)) < p_sim).astype(float)
print(f"planted effect: mean {true_tau.mean():+.3f}, SD {true_tau.std():.3f}, range [{true_tau.min():+.3f}, {true_tau.max():+.3f}]")

tr_i, te_i = train_test_split(np.arange(len(ts)), test_size=0.5, stratify=ts, random_state=SEED)
rows = []
sim_null_rng = np.random.default_rng(SEED + 7)
null_auuc = [evaluate(y_sim[te_i], ts[te_i], sim_null_rng.normal(size=len(te_i)), sim_null_rng)[0] for _ in range(300)]
for name in LEARNERS:
    m = make_learner(name, SEED)
    m.fit(y_sim[tr_i], ts[tr_i], X=Xs.iloc[tr_i])
    tau_hat = np.ravel(m.effect(Xs.iloc[te_i]))
    a, top, ate = evaluate(y_sim[te_i], ts[te_i], tau_hat, np.random.default_rng(SEED))
    rows.append({"model": name, "AUUC": a, "top-20% effect": top, "corr with true effect": np.corrcoef(tau_hat, true_tau[te_i])[0, 1]})
best_rule, rule_fn = rule_scores(Xs.iloc[tr_i], y_sim[tr_i], ts[tr_i], np.random.default_rng(SEED))
a, top, ate = evaluate(y_sim[te_i], ts[te_i], rule_fn(Xs.iloc[te_i]), np.random.default_rng(SEED))
rows.append({"model": f"Rule ({best_rule})", "AUUC": a, "top-20% effect": top, "corr with true effect": np.corrcoef(rule_fn(Xs.iloc[te_i]), true_tau[te_i])[0, 1]})
check2 = pd.DataFrame(rows).set_index("model")
display(check2.round(4))
print(f"overall effect in the simulated test half: {ate:+.4f}")
print(f"AUUC under random ranking (null): 97.5th percentile = {np.percentile(null_auuc, 97.5):.5f}")

## 3 · Fit and compare on held-out customers

For each email, outcome and random split, every learner is trained on one half and scored on the other. Everything below is computed on the held-out halves only. The first split is kept in detail for the bootstrap, permutation and calibration analyses; all five feed the stability table.

In [ ]:
records, kept = [], {}
cf_importance = {}
for arm in ARMS:
    d = data[arm]
    for r in range(N_SPLITS):
        tr_i, te_i = train_test_split(np.arange(len(d["t"])), test_size=0.5, stratify=d["t"], random_state=SEED + r)
        Xtr, Xte = d["X"].iloc[tr_i], d["X"].iloc[te_i]
        for o in OUTCOMES:
            ytr, yte = d["y"][o][tr_i], d["y"][o][te_i]
            ttr, tte = d["t"][tr_i], d["t"][te_i]
            rng = np.random.default_rng(SEED + 1000 * r)
            scores = {}
            for name in LEARNERS:
                m = make_learner(name, SEED + r)
                m.fit(ytr, ttr, X=Xtr)
                scores[name] = np.ravel(m.effect(Xte))
                if name == "Causal forest" and r == 0 and o == "visit":
                    cf_importance[arm] = pd.Series(m.feature_importances_, index=FEATURES)
            rule_name, rule_fn = rule_scores(Xtr, ytr, ttr, rng)
            scores["Rule"] = rule_fn(Xte)
            for name, s in scores.items():
                a, top, ate = evaluate(yte, tte, s, np.random.default_rng(SEED + 1000 * r + 1))
                records.append({"arm": arm, "outcome": o, "split": r, "model": name, "auuc": a, "top20_effect": top,
                                "ate_test": ate, "rule_used": rule_name if name == "Rule" else ""})
            if r == 0:
                kept[(arm, o)] = {"y": yte, "t": tte, "scores": scores, "X": Xte, "rule": rule_name}
    print(f"done: {arm}")

results = pd.DataFrame(records)

In [ ]:
summary = (results.groupby(["arm", "outcome", "model"])
           .agg(auuc_mean=("auuc", "mean"), auuc_sd=("auuc", "std"), top20_mean=("top20_effect", "mean"), ate_mean=("ate_test", "mean"))
           .reset_index())
rule_auuc = results[results["model"] == "Rule"].set_index(["arm", "outcome", "split"])["auuc"]
beats = results[results["model"] != "Rule"].copy()
beats["rule_auuc"] = [rule_auuc[(a, o, s)] for a, o, s in zip(beats["arm"], beats["outcome"], beats["split"])]
beats["beats_rule"] = beats["auuc"] > beats["rule_auuc"]
share = beats.groupby(["arm", "outcome", "model"])["beats_rule"].mean().rename("splits_beating_rule")
summary = summary.merge(share.reset_index(), on=["arm", "outcome", "model"], how="left")
summary["model"] = pd.Categorical(summary["model"], MODELS, ordered=True)
summary = summary.sort_values(["arm", "outcome", "model"]).reset_index(drop=True)
print(f"AUUC over {N_SPLITS} random 50/50 splits (mean, SD). Units: extra outcome per customer relative to random targeting.")
for arm in ARMS:
    print(f"\n=== {arm} ===")
    display(summary[summary["arm"] == arm].drop(columns="arm").round({"auuc_mean": 5, "auuc_sd": 5, "top20_mean": 4, "ate_mean": 4, "splits_beating_rule": 2}))
print("\nRule chosen on training data, by split (visit):")
display(results[(results["model"] == "Rule") & (results["outcome"] == "visit")].pivot(index="split", columns="arm", values="rule_used"))

### Uncertainty on the held-out results, and a permutation null

The standard deviations above reflect different train/test splits of the same data. They do not show how much a single held-out half could differ from the truth, which is the larger source of noise here. For the first split, two further analyses use the held-out half only:

- **Bootstrap** (stratified by arm, 1,000 replicates, same resamples for every model so differences are paired) gives 95% intervals for AUUC and for the top-20% effect, and for each model's AUUC **minus the rule's**.
- **Permutation null:** ranking the same customers at random 1,000 times shows what AUUC looks like when the ranking carries no information. The p-value is the share of random rankings with an AUUC at least as large as the model's.

In [ ]:
boot_rows, null_store = [], {}
for (arm, o), k in kept.items():
    y, t = k["y"], k["t"]
    n_t, n_c = int(t.sum()), int((1 - t).sum())
    idx_t, idx_c = np.where(t == 1)[0], np.where(t == 0)[0]
    brng = np.random.default_rng(SEED + 99)
    W = np.zeros((N_BOOT, len(y)))
    for b in range(N_BOOT):                                        # stratified bootstrap weights, shared by all models
        W[b, idx_t] = np.bincount(brng.integers(0, n_t, n_t), minlength=n_t)
        W[b, idx_c] = np.bincount(brng.integers(0, n_c, n_c), minlength=n_c)
    # permutation null: random rankings of the same customers
    prng = np.random.default_rng(SEED + 5)
    null = np.empty(N_BOOT)
    for b in range(N_BOOT):
        perm = prng.permutation(len(y))
        phi, G = gain_curve(y[perm], t[perm])
        null[b] = auuc(phi, G)
    null_store[(arm, o)] = null

    per_model = {}
    for name, s in k["scores"].items():
        order = np.lexsort((np.random.default_rng(SEED + 1).random(len(y)), -s))
        ys, ts_ = y[order], t[order]
        Ws = W[:, order]
        a_b, top_b = np.empty(N_BOOT), np.empty(N_BOOT)
        for b in range(N_BOOT):
            phi, G = gain_curve(ys, ts_, Ws[b])
            a_b[b], top_b[b] = auuc(phi, G), top_effect(phi, G)
        phi, G = gain_curve(ys, ts_)
        per_model[name] = {"auuc": auuc(phi, G), "top": top_effect(phi, G), "ate": G[-1], "a_b": a_b, "top_b": top_b}
    for name, m in per_model.items():
        diff_b = m["a_b"] - per_model["Rule"]["a_b"]
        boot_rows.append({"arm": arm, "outcome": o, "model": name, "auuc": m["auuc"],
                          "auuc_lo": np.percentile(m["a_b"], 2.5), "auuc_hi": np.percentile(m["a_b"], 97.5),
                          "perm_p": (1 + (null >= m["auuc"]).sum()) / (1 + N_BOOT),
                          "diff_vs_rule": m["auuc"] - per_model["Rule"]["auuc"],
                          "diff_lo": np.percentile(diff_b, 2.5), "diff_hi": np.percentile(diff_b, 97.5),
                          "top20_effect": m["top"], "top20_lo": np.percentile(m["top_b"], 2.5), "top20_hi": np.percentile(m["top_b"], 97.5),
                          "ate_test": m["ate"]})
boot = pd.DataFrame(boot_rows)
boot.loc[boot["model"] == "Rule", ["diff_vs_rule", "diff_lo", "diff_hi"]] = np.nan
boot["model"] = pd.Categorical(boot["model"], MODELS, ordered=True)
boot = boot.sort_values(["arm", "outcome", "model"]).reset_index(drop=True)
for arm in ARMS:
    print(f"\n=== {arm}: first split, held-out half ===")
    display(boot[boot["arm"] == arm].drop(columns="arm").round({"auuc": 5, "auuc_lo": 5, "auuc_hi": 5, "perm_p": 3, "diff_vs_rule": 5, "diff_lo": 5, "diff_hi": 5,
                                                                  "top20_effect": 4, "top20_lo": 4, "top20_hi": 4, "ate_test": 4}))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5))
for i, arm in enumerate(ARMS):
    for j, o in enumerate(OUTCOMES):
        ax = axes[i, j]
        b = boot[(boot["arm"] == arm) & (boot["outcome"] == o)].reset_index(drop=True)
        null = null_store[(arm, o)]
        ax.axvspan(*np.percentile(null, [2.5, 97.5]), color="grey", alpha=0.2, label="random ranking (95%)")
        ax.errorbar(b["auuc"], range(len(b)), xerr=[b["auuc"] - b["auuc_lo"], b["auuc_hi"] - b["auuc"]], fmt="o", capsize=3, ms=4)
        ax.set_yticks(range(len(b)), b["model"].astype(str) if j == 0 else [""] * len(b))
        ax.invert_yaxis()
        ax.axvline(0, color="grey", lw=0.8)
        ax.set_title(f"{ARM_TITLE[arm]}: {o}", fontsize=10)
        if i == 1:
            ax.set_xlabel("AUUC (extra outcome per customer vs random)")
        ax.xaxis.set_major_locator(MaxNLocator(4))
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=8, loc="lower center", ncol=2)
plt.suptitle("Held-out AUUC with 95% bootstrap intervals; grey band = what random ranking gives", y=1.0)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

### Gain curves for visit (first split, held-out half)

In [ ]:
colors = {"S-learner": "C0", "T-learner": "C1", "X-learner": "C2", "Causal forest": "C3", "Rule": "C7"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, arm in zip(axes, ARMS):
    k = kept[(arm, "visit")]
    for name, s in k["scores"].items():
        ys, ts_ = ranked_arrays(k["y"], k["t"], s, np.random.default_rng(SEED + 1))
        phi, G = gain_curve(ys, ts_)
        ax.plot(phi, G, color=colors[name], lw=1.4, ls="--" if name == "Rule" else "-", label=name if name != "Rule" else f"Rule: {k['rule']}")
    ax.plot([0, 1], [0, G[-1]], color="black", lw=1, label="random targeting")
    ax.set_xlabel("share of customers emailed (ranked by predicted effect)")
    ax.set_title(f"{ARM_TITLE[arm]}: visit")
axes[0].set_ylabel("extra visits per customer in the population")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 4 · Calibration by decile

A model can rank customers well and still give effect sizes that are too large or too small. Held-out customers are sorted by predicted effect into ten equal groups (decile 1 = highest predicted effect). In each group the **observed** effect (emailed minus not-emailed visit rate) is compared with the average **predicted** effect. Points on the diagonal mean the predictions are well calibrated. A slope below the diagonal means the predictions are more spread out than the real differences.

In [ ]:
decile_rows = []
fig, axes = plt.subplots(2, 4, figsize=(13, 6.2), sharex=False, sharey=False)
for i, arm in enumerate(ARMS):
    k = kept[(arm, "visit")]
    for j, name in enumerate(LEARNERS):
        s = k["scores"][name]
        order = np.argsort(-s, kind="mergesort")
        groups = np.array_split(order, 10)
        pred, obs, se = [], [], []
        for g in groups:
            yg, tg = k["y"][g], k["t"][g]
            m1, m0 = yg[tg == 1].mean(), yg[tg == 0].mean()
            v = yg[tg == 1].var(ddof=1) / (tg == 1).sum() + yg[tg == 0].var(ddof=1) / (tg == 0).sum()
            pred.append(s[g].mean()); obs.append(m1 - m0); se.append(np.sqrt(v))
        pred, obs, se = map(np.array, (pred, obs, se))
        for dec in range(10):
            decile_rows.append({"arm": arm, "model": name, "decile": dec + 1, "predicted": pred[dec], "observed": obs[dec], "se": se[dec]})
        ax = axes[i, j]
        ax.errorbar(100 * pred, 100 * obs, yerr=100 * 1.96 * se, fmt="o", capsize=2, ms=3.5)
        lims = [100 * min(pred.min(), obs.min() - 1.96 * se.max()), 100 * max(pred.max(), obs.max() + 1.96 * se.max())]
        ax.plot(lims, lims, color="grey", lw=0.8, ls="--")
        ax.set_title(f"{name}" if i == 0 else "", fontsize=10)
        if j == 0:
            ax.set_ylabel(f"{ARM_TITLE[arm]}\nobserved effect (pp)")
        if i == 1:
            ax.set_xlabel("mean predicted effect (pp)")
plt.suptitle("Visit: observed vs predicted effect by decile of predicted effect (held-out, first split)", y=1.0)
plt.tight_layout()
plt.show()

deciles = pd.DataFrame(decile_rows)
cal_rows = []
for (arm, name), g in deciles.groupby(["arm", "model"]):
    top, bot = g[g["decile"] == 1].iloc[0], g[g["decile"] == 10].iloc[0]
    diff, se_d = top["observed"] - bot["observed"], np.sqrt(top["se"] ** 2 + bot["se"] ** 2)
    slope = np.polyfit(g["predicted"], g["observed"], 1)[0]
    cal_rows.append({"arm": arm, "model": name, "observed_top_decile": top["observed"], "observed_bottom_decile": bot["observed"],
                     "top_minus_bottom": diff, "ci_low": diff - 1.96 * se_d, "ci_high": diff + 1.96 * se_d,
                     "rank_corr_pred_vs_obs": stats.spearmanr(g["predicted"], g["observed"])[0], "calibration_slope": slope})
calibration = pd.DataFrame(cal_rows)
display(calibration.round(4))

## 5 · What does the causal forest rely on?

Feature importance from the causal forest (first split, visit) shows which variables the forest splits on most when it looks for differences in the email effect. This describes the forest's behaviour. It is not a statement that those variables cause response differences, and with weak heterogeneity the importances can reflect noise.

In [ ]:
imp = pd.DataFrame(cf_importance)
imp = imp.loc[imp.mean(axis=1).sort_values().index]
ax = imp.rename(columns=ARM_TITLE).plot.barh(figsize=(7, 4.5), width=0.75)
ax.set_xlabel("share of the forest's splits (depth-weighted)")
ax.set_title("Causal forest feature importance (visit)")
plt.tight_layout()
plt.show()
display(imp.sort_values("mens_email", ascending=False).round(3))

## 6 · Summary of results

In [ ]:
print("Heterogeneity tests (visit, pre-specified subgroups, Holm across 14 tests):")
print("  significant after Holm:", tests.loc[tests["significant_holm"], ["arm", "variable"]].values.tolist() or "none")
print(f"  smallest Holm-adjusted p: {tests['p_holm'].min():.4f}")
print("Joint interaction tests (3 outcomes x 2 emails, Holm):")
print("  significant after Holm:", joint.loc[joint["significant_holm"], ["arm", "outcome"]].values.tolist() or "none")
print(f"  smallest Holm-adjusted p: {joint['p_holm'].min():.4f}\n")

best = (boot[boot["model"] != "Rule"].sort_values("auuc", ascending=False).groupby(["arm", "outcome"]).head(1)
        .sort_values(["arm", "outcome"]))
print("Best learner per email and outcome on the first held-out half (not a fair estimate of its true performance, since it was picked after looking):")
for _, r in best.iterrows():
    print(f"  {r['arm']:13s} {r['outcome']:10s} {str(r['model']):14s} AUUC {r['auuc']:+.5f} [{r['auuc_lo']:+.5f}, {r['auuc_hi']:+.5f}]  permutation p = {r['perm_p']:.3f}  "
          f"vs rule {r['diff_vs_rule']:+.5f} [{r['diff_lo']:+.5f}, {r['diff_hi']:+.5f}]")

tests.to_csv(TABLE_DIR / "02_subgroup_interaction_tests_visit.csv", index=False)
effects.to_csv(TABLE_DIR / "02_subgroup_effects_visit.csv", index=False)
joint.to_csv(TABLE_DIR / "02_joint_interaction_tests.csv", index=False)
summary.to_csv(TABLE_DIR / "02_model_comparison_repeated_splits.csv", index=False)
boot.to_csv(TABLE_DIR / "02_model_comparison_bootstrap_first_split.csv", index=False)
calibration.to_csv(TABLE_DIR / "02_decile_calibration_visit.csv", index=False)
check2.to_csv(TABLE_DIR / "02_planted_effect_check.csv")
print(f"\nTables written to {TABLE_DIR.resolve()}")

## Interpretation and limitations

**What the evidence supports**

1. **Women's email: the effect varies, and models can find it.** Across five repeated 50/50 splits, every model beat the simple rule on visits, conversion and spend in all five splits. On the first held-out half the visit AUUC is about +0.007 to +0.008 extra visits per customer (95% intervals exclude zero; permutation p = 0.001), against a population-wide effect of about +0.048. The top 20% by predicted effect shows a visit effect of roughly +7.7 to +9.5 points against +4.8 for emailing everyone. Predicted and observed effects agree by decile (rank correlation 0.76 to 0.95), and the top decile is about 5 to 7 points above the bottom decile. The forest's splits and the subgroup tests point the same way: past purchases of women's and men's merchandise carry the signal. Learners differ little from each other, so which one to use is not the interesting question here.
2. **Men's email: no usable ranking was found.** The joint interaction test for visits is significant, but no model beats random targeting convincingly (first-split permutation p = 0.08 to 0.39 for visits) and none beats the simple rule. Decile calibration is poor (calibration slopes 0.12 to 0.42, top-minus-bottom intervals include zero). The forest's top-ranked feature is the continuous `history` variable, which is the pattern one expects from noise-fitting. The accurate summary is that heterogeneity may exist but this sample cannot rank customers by it.
3. **Conversion and spend are noisier.** Conversion is about 0.3% to 1.2%, and spend is heavy-tailed, so intervals are wide. Women's spend and conversion show some signal; men's show none that survives the uncertainty.
4. **Check on the method.** With a planted effect of known form (size similar to the men's email effect), the same pipeline recovers a ranking with correlation 0.77 to 0.87 with the truth. So the weak result for the men's email is not a sign that the pipeline cannot find structure of that size, but it is not proof that no structure exists.

**Limits**

- About 30 permutation p-values are reported without a multiplicity correction. Read the pattern across them, not any single one.
- The "best learner" table picks winners after looking at the held-out half, so it overstates their performance. The repeated-split table is the fairer comparison.
- The standard deviations across repeated splits understate uncertainty about how a model would perform on new customers, because the splits reuse the same 43,000 customers.
- Two-week window; customers who bought within the previous 12 months; the two emails differ in content as well as audience, so a "women's email effect" mixes the two.
- Revenue is gross. Whether targeting beats emailing everyone depends on email cost and margin, which notebook 04 handles with explicit assumptions.